### 🧩 Step: Merge and Clip Historical Wetland Layers for NDVI Analysis

This step prepares the **historical wetland polygons** provided by Tyler for integration with the Landsat-based NDVI workflow.  
The raw data are distributed as individual **HUC8-level shapefiles** covering the broader U.S. Prairie Pothole Region.  
Because the NDVI analysis focuses on the **Des Moines Lobe (DML)**, the script merges, cleans, simplifies, and clips these layers to the DML boundary.

#### **What the script does**

1. **Merge HUC8 shapefiles**  
   - Automatically identifies all shapefiles matching `*_sinkvect_g3_d005.shp` within  
     the directory configured via the `DML_PPR_VECT_ROOT` environment variable.  
   - Combines them into a single GeoDataFrame, preserving HUC8 identifiers for traceability.

2. **Standardize coordinate systems and geometry validity**  
   - Reprojects all layers to **EPSG:5070 (NAD83 / Conus Albers)**, ensuring alignment with the NDVI raster grid.  
   - Fixes any invalid geometries using `shapely.make_valid()` or a `buffer(0)` fallback.

3. **Simplify and optimize for NDVI sampling performance**  
   - Reduces vertex density using precision snapping and topology-preserving simplification (`tolerance = 15 m`).  
   - Removes very small polygons (< 1,000 m²) to eliminate slivers and speed up raster sampling.  
   - Optionally allows stronger simplification or dissolve by HUC8 for extremely large datasets.

4. **Clip to the Des Moines Lobe boundary**  
   - Loads the boundary shapefile:  
     `<DML_GIS_DATA_ROOT>/Shapefiles/dml_boundaries_5070.shp` (configured via `DML_GIS_DATA_ROOT`).  
   - Uses `geopandas.clip()` to retain only wetlands that fall within the DML extent.

5. **Finalize and export**  
   - Performs a second simplification pass post-clip for efficiency.  
   - Converts multipart geometries to singlepart features.  
   - Exports both a **Shapefile** and a **GeoPackage**:  
     - `DML_wetlands_from_Tyler_clipped_5070.shp`  
     - `DML_wetlands_from_Tyler_clipped_5070.gpkg`

#### **Purpose**

This script produces a clean, lightweight, and spatially consistent dataset of historical wetlands confined to the **Des Moines Lobe**, ready for efficient NDVI time-series extraction and analysis.  
By simplifying geometry and removing redundant detail, it ensures the subsequent NDVI sampling step will run smoothly without compromising spatial integrity.


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Clip Tyler's historical wetland HUC8 shapefiles to the Des Moines Lobe (DML),
lighten geometry for fast NDVI sampling, and write polygon-only outputs.

Speed tweaks:
- Read with bbox limited to the DML (tuple, not ndarray)
- Clip each HUC8 individually, then simplify/clean
- Log progress at each step

Requires: geopandas, shapely>=2.0, pandas, pyproj, pyogrio (for fast I/O)
"""

from pathlib import Path
import sys
import warnings
import re
import time
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import os

# Prefer pyogrio if available (faster I/O)
try:
    gpd.options.io_engine = "pyogrio"
except Exception:
    pass

# ---------------------------
# USER SETTINGS
# ---------------------------
SRC_DIR = Path(os.environ.get("DML_PPR_VECT_ROOT", "./ppr_vector_data"))
DML_PATH = Path(os.environ.get("DML_GIS_DATA_ROOT", "./gis_data")) / "Shapefiles/dml_boundaries_5070.shp"

OUT_DIR = Path(os.environ.get("DML_NDVI_DATA_ROOT", "./ndvi_wetlands_data")) / "working"
OUT_BASENAME = "DML_wetlands_from_Tyler"

# Which files to ingest (adjust if Tyler changes names)
PATTERNS = ["*_sinkvect_g3_d005.shp", "*sinkvect*g3*d005.shp"]

# Geometry lightening controls (EPSG:5070 units = meters)
GRID_SIZE_M              = 1.0    # coordinate precision grid; set 0 to disable
PRESIMPLIFY_TOL_M        = 0.0    # mild simplify before clip; usually keep 0 for speed
POSTCLIP_SIMPLIFY_TOL_M  = 12.0   # main simplify after clip (10–25 is sensible)
DROP_SMALL_SLIVERS_M2    = 1000   # drop polygons smaller than this area

# Optional dissolve (usually False; keeps per-wetland features)
DO_DISSOLVE_BY_HUC8 = False

# ---------------------------
# Helpers
# ---------------------------

def log(msg: str):
    print(f"[DML-CLIP] {msg}", flush=True)

def find_shapefiles(folder: Path, patterns):
    files = []
    for pat in patterns:
        files.extend(folder.glob(pat))
    # Deduplicate while preserving order
    seen = set()
    uniq = []
    for f in files:
        if f not in seen:
            uniq.append(f)
            seen.add(f)
    return uniq

def infer_huc8_from_name(path: Path):
    m = re.search(r"(\d{8})", path.stem)
    return m.group(1) if m else None

def safe_to_crs(gdf: gpd.GeoDataFrame, epsg: int) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        warnings.warn("Input layer has no CRS; assuming EPSG:5070. Adjust if incorrect.")
        gdf = gdf.set_crs(epsg=epsg)
        return gdf
    return gdf.to_crs(epsg=epsg)

def fast_make_valid(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Cheaper validity fix after clip/simplify (smaller geoms)."""
    try:
        import shapely
        has_make_valid = hasattr(shapely, "make_valid")
    except Exception:
        has_make_valid = False
    if has_make_valid:
        from shapely import make_valid
        gdf["geometry"] = gdf.geometry.apply(make_valid)
    else:
        gdf["geometry"] = gdf.buffer(0)
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
    return gdf

def keep_area_geoms(gdf):
    # keep only Polygons/MultiPolygons
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    # drop zero-area remnants
    gdf = gdf[gdf.geometry.area > 0].copy()
    return gdf

def promote_to_multipolygon(gdf):
    def _promote(g):
        if isinstance(g, Polygon):
            return MultiPolygon([g])
        return g
    gdf["geometry"] = gdf.geometry.apply(_promote)
    return gdf

def set_precision_opt(geom, grid):
    if grid and grid > 0:
        try:
            from shapely import set_precision
            return set_precision(geom, grid)
        except Exception:
            return geom
    return geom

def simplify_opt(gdf, tol):
    if tol and tol > 0:
        gdf["geometry"] = gdf.geometry.simplify(tol, preserve_topology=True)
    return gdf

def drop_small_slivers(gdf, area_thresh_m2):
    if area_thresh_m2 and area_thresh_m2 > 0:
        before = len(gdf)
        gdf = gdf[gdf.geometry.area >= area_thresh_m2].copy()
        dropped = before - len(gdf)
        if dropped > 0:
            log(f"Dropped {dropped} tiny polygons (< {area_thresh_m2} m²).")
    return gdf

# ---------------------------
# Main
# ---------------------------

def main():
    t0 = time.time()

    if not SRC_DIR.exists():
        log(f"ERROR: Source directory not found: {SRC_DIR}")
        sys.exit(1)
    if not DML_PATH.exists():
        log(f"ERROR: DML boundary not found: {DML_PATH}")
        sys.exit(1)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load DML, make single polygon, get bbox in 5070
    log("Loading DML boundary ...")
    dml = gpd.read_file(DML_PATH)
    dml = safe_to_crs(dml, 5070)
    if len(dml) > 1:
        dml = dml.dissolve().explode(index_parts=False).reset_index(drop=True)
    dml_poly = dml.iloc[0:1].copy()
    # cast to plain Python tuple for bbox (not ndarray)
    dml_bounds = tuple(map(float, dml_poly.total_bounds))  # (minx, miny, maxx, maxy)

    # Discover candidate shapefiles
    shp_files = find_shapefiles(SRC_DIR, PATTERNS)
    if not shp_files:
        log("ERROR: No shapefiles matched the patterns. Check PATTERNS or folder.")
        sys.exit(1)
    log(f"Found {len(shp_files)} candidate HUC8 shapefiles.")

    # Process each HUC8 individually (read → bbox filter → clip → clean)
    clipped_parts = []
    kept_counts = 0
    skipped_empty = 0

    for i, f in enumerate(shp_files, start=1):
        log(f"[{i}/{len(shp_files)}] Reading (bbox=DML) → {f.name}")
        try:
            # Primary: bbox read (fast)
            try:
                gdf = gpd.read_file(f, bbox=dml_bounds)
            except TypeError:
                # Some engines require list or tuple; we already cast, but just in case:
                gdf = gpd.read_file(f, bbox=(dml_bounds[0], dml_bounds[1], dml_bounds[2], dml_bounds[3]))
            except Exception as e_bbox:
                log(f"  bbox read failed ({e_bbox}); trying mask read ...")
                gdf = gpd.read_file(f, mask=dml_poly.geometry.iloc[0])

            if gdf.empty:
                skipped_empty += 1
                continue

            gdf = safe_to_crs(gdf, 5070)

            # Optional mild pre-simplify (usually 0 for speed)
            if GRID_SIZE_M and GRID_SIZE_M > 0:
                gdf["geometry"] = gdf.geometry.apply(lambda g: set_precision_opt(g, GRID_SIZE_M))
            if PRESIMPLIFY_TOL_M and PRESIMPLIFY_TOL_M > 0:
                gdf = simplify_opt(gdf, PRESIMPLIFY_TOL_M)

            # Clip to DML polygon
            gdf = gpd.clip(gdf, dml_poly)
            if gdf.empty:
                continue

            # Post-clip: clean & lighten
            gdf = fast_make_valid(gdf)
            gdf = keep_area_geoms(gdf)
            gdf = drop_small_slivers(gdf, DROP_SMALL_SLIVERS_M2)
            gdf = simplify_opt(gdf, POSTCLIP_SIMPLIFY_TOL_M)
            gdf = fast_make_valid(gdf)

            # Track HUC8 from filename
            huc8 = infer_huc8_from_name(f)
            if huc8 is not None and "HUC8_FILE" not in gdf.columns:
                gdf["HUC8_FILE"] = huc8

            if not gdf.empty:
                clipped_parts.append(gdf)
                kept_counts += len(gdf)

        except Exception as e:
            log(f"WARNING: Skipped {f.name} due to error: {e}")
            continue

    if not clipped_parts:
        log("No features remained after bbox/mask+clip. Check inputs and DML boundary.")
        sys.exit(1)

    # Merge all clipped pieces
    log(f"Concatenating {len(clipped_parts)} clipped subsets ...")
    merged = gpd.GeoDataFrame(pd.concat(clipped_parts, ignore_index=True), crs=5070)

    # Optional dissolve by HUC8 to reduce vertex counts further
    if DO_DISSOLVE_BY_HUC8 and "HUC8_FILE" in merged.columns:
        log("Dissolving by HUC8_FILE ...")
        merged = merged.dissolve(by="HUC8_FILE", as_index=False)
        merged = fast_make_valid(merged)

    # Final clean: explode multipart, enforce polygon-only multipolygons
    log("Final clean / explode ...")
    merged = merged.explode(index_parts=False).reset_index(drop=True)
    merged = keep_area_geoms(merged)
    merged = promote_to_multipolygon(merged)
    merged = fast_make_valid(merged)

    # Outputs
    shp_out  = OUT_DIR / f"{OUT_BASENAME}_clipped_5070.shp"
    gpkg_out = OUT_DIR / f"{OUT_BASENAME}_clipped_5070.gpkg"

    log(f"Writing Shapefile → {shp_out}")
    merged.to_file(
        shp_out,
        driver="ESRI Shapefile",
        promote_to_multi=True,
        geometry_type="MultiPolygon",
    )

    log(f"Writing GeoPackage → {gpkg_out}")
    merged.to_file(
        gpkg_out,
        layer="dml_wetlands",
        driver="GPKG",
        promote_to_multi=True,
        geometry_type="MultiPolygon",
    )

    # Stats
    total_feats = len(merged)
    total_area_km2 = merged.geometry.area.sum() / 1e6
    t1 = time.time()
    log(f"Done. Features kept: {total_feats:,} | Total area: {total_area_km2:,.2f} km²")
    log(f"Skipped empty-after-bbox files: {skipped_empty}")
    log(f"Elapsed: {t1 - t0:,.1f} s")
    log(f"Outputs:\n  - {shp_out}\n  - {gpkg_out}")

if __name__ == "__main__":
    main()


[DML-CLIP] Loading DML boundary ...
[DML-CLIP] Found 134 candidate HUC8 shapefiles.
[DML-CLIP] [1/134] Reading (bbox=DML) → HUC8_07010101_sinkvect_g3_d005.shp
[DML-CLIP] [2/134] Reading (bbox=DML) → HUC8_07010102_sinkvect_g3_d005.shp
[DML-CLIP] [3/134] Reading (bbox=DML) → HUC8_07010103_sinkvect_g3_d005.shp
[DML-CLIP] [4/134] Reading (bbox=DML) → HUC8_07010104_sinkvect_g3_d005.shp
[DML-CLIP] [5/134] Reading (bbox=DML) → HUC8_07010105_sinkvect_g3_d005.shp
[DML-CLIP] [6/134] Reading (bbox=DML) → HUC8_07010106_sinkvect_g3_d005.shp
[DML-CLIP] [7/134] Reading (bbox=DML) → HUC8_07010107_sinkvect_g3_d005.shp
[DML-CLIP] [8/134] Reading (bbox=DML) → HUC8_07010108_sinkvect_g3_d005.shp
[DML-CLIP] [9/134] Reading (bbox=DML) → HUC8_07010201_sinkvect_g3_d005.shp
[DML-CLIP] [10/134] Reading (bbox=DML) → HUC8_07010202_sinkvect_g3_d005.shp
[DML-CLIP] [11/134] Reading (bbox=DML) → HUC8_07010203_sinkvect_g3_d005.shp
[DML-CLIP] [12/134] Reading (bbox=DML) → HUC8_07010204_sinkvect_g3_d005.shp
[DML-CLIP] [1

/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


[DML-CLIP] [25/134] Reading (bbox=DML) → HUC8_07020010_sinkvect_g3_d005.shp


/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/constructive.py:863: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


[DML-CLIP] [26/134] Reading (bbox=DML) → HUC8_07020011_sinkvect_g3_d005.shp
[DML-CLIP] [27/134] Reading (bbox=DML) → HUC8_07020012_sinkvect_g3_d005.shp
[DML-CLIP] [28/134] Reading (bbox=DML) → HUC8_07030001_sinkvect_g3_d005.shp
[DML-CLIP] [29/134] Reading (bbox=DML) → HUC8_07030002_sinkvect_g3_d005.shp
[DML-CLIP] [30/134] Reading (bbox=DML) → HUC8_07030003_sinkvect_g3_d005.shp
[DML-CLIP] [31/134] Reading (bbox=DML) → HUC8_07030004_sinkvect_g3_d005.shp
[DML-CLIP] [32/134] Reading (bbox=DML) → HUC8_07030005_sinkvect_g3_d005.shp
[DML-CLIP] [33/134] Reading (bbox=DML) → HUC8_07040001_sinkvect_g3_d005.shp
[DML-CLIP] [34/134] Reading (bbox=DML) → HUC8_07040002_sinkvect_g3_d005.shp
[DML-CLIP] [35/134] Reading (bbox=DML) → HUC8_07040003_sinkvect_g3_d005.shp
[DML-CLIP] [36/134] Reading (bbox=DML) → HUC8_07040004_sinkvect_g3_d005.shp
[DML-CLIP] [37/134] Reading (bbox=DML) → HUC8_07040005_sinkvect_g3_d005.shp
[DML-CLIP] [38/134] Reading (bbox=DML) → HUC8_07040006_sinkvect_g3_d005.shp
[DML-CLIP] [

...

[DML-CLIP] Dropped 3089 tiny polygons (< 1000 m²).
[DML-CLIP] [89/134] Reading (bbox=DML) → HUC8_07100008_sinkvect_g3_d005.shp


/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)
/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/constructive.py:863: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


[DML-CLIP] Dropped 1206 tiny polygons (< 1000 m²).
[DML-CLIP] [90/134] Reading (bbox=DML) → HUC8_07100009_sinkvect_g3_d005.shp


/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)
/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/constructive.py:863: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


[DML-CLIP] [91/134] Reading (bbox=DML) → HUC8_07110001_sinkvect_g3_d005.shp
[DML-CLIP] [92/134] Reading (bbox=DML) → HUC8_07110002_sinkvect_g3_d005.shp
[DML-CLIP] [93/134] Reading (bbox=DML) → HUC8_07110003_sinkvect_g3_d005.shp
[DML-CLIP] [94/134] Reading (bbox=DML) → HUC8_07110004_sinkvect_g3_d005.shp
[DML-CLIP] [95/134] Reading (bbox=DML) → HUC8_07110005_sinkvect_g3_d005.shp
[DML-CLIP] [96/134] Reading (bbox=DML) → HUC8_07110006_sinkvect_g3_d005.shp
[DML-CLIP] [97/134] Reading (bbox=DML) → HUC8_07110007_sinkvect_g3_d005.shp
[DML-CLIP] [98/134] Reading (bbox=DML) → HUC8_07110008_sinkvect_g3_d005.shp
[DML-CLIP] [99/134] Reading (bbox=DML) → HUC8_07110009_sinkvect_g3_d005.shp
[DML-CLIP] [100/134] Reading (bbox=DML) → HUC8_07120001_sinkvect_g3_d005.shp
[DML-CLIP] [101/134] Reading (bbox=DML) → HUC8_07120002_sinkvect_g3_d005.shp
[DML-CLIP] [102/134] Reading (bbox=DML) → HUC8_07120003_sinkvect_g3_d005.shp
[DML-CLIP] [103/134] Reading (bbox=DML) → HUC8_07120004_sinkvect_g3_d005.shp
[DML-CLI

In [ ]:
# filter wetlands by size

In [10]:
#!/usr/bin/env python3
# filter_wetlands_by_area.py
# Keep only wetlands <= 647,500 m² (1 sq mi / 4)

from pathlib import Path
import sys
import geopandas as gpd
import os

# -------------------------------
# USER PATHS
# -------------------------------
BASE = Path(os.environ.get("DML_NDVI_DATA_ROOT", "./ndvi_wetlands_data"))

# Input: the shapefile you created yesterday
IN_SHP = BASE / "working" / "DML_wetlands_from_Tyler_clipped_5070.shp"

# Output folder + filenames
OUT_DIR = BASE / "Historical_Wetlands_FILT"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SHP = OUT_DIR / "Historical_Wetlands_TYLER_max647500m2_5070.shp"  # filtered
OUT_GPKG = OUT_DIR / "Historical_Wetlands_TYLER_max647500m2_5070.gpkg"  # optional

# -------------------------------
# SETTINGS
# -------------------------------
MAX_WETLAND_AREA_M2 = 647_500
TARGET_CRS = "EPSG:5070"  # projected, meters

def main():
    if not IN_SHP.exists():
        print(f"[ERROR] Input shapefile not found: {IN_SHP}")
        sys.exit(1)

    print(f"[LOAD] {IN_SHP}")
    gdf = gpd.read_file(IN_SHP)

    # Reproject if needed
    if gdf.crs is None:
        print("[WARN] Layer has no CRS; assuming EPSG:5070. Adjust if this is wrong.")
        gdf.set_crs(TARGET_CRS, inplace=True)
    elif gdf.crs.to_string() != TARGET_CRS:
        print(f"[REPROJECT] {gdf.crs.to_string()} → {TARGET_CRS}")
        gdf = gdf.to_crs(TARGET_CRS)

    # Fix invalid geometries (only those that need it)
    if not gdf.geometry.is_valid.all():
        n_invalid = (~gdf.geometry.is_valid).sum()
        print(f"[FIX] Repairing {n_invalid} invalid geometries with buffer(0)")
        gdf.loc[~gdf.geometry.is_valid, "geometry"] = gdf.loc[~gdf.geometry.is_valid, "geometry"].buffer(0)

    # Compute area and filter
    gdf["area_m2"] = gdf.geometry.area
    n_before = len(gdf)
    kept = gdf[gdf["area_m2"] <= MAX_WETLAND_AREA_M2].copy()
    n_after = len(kept)
    n_removed = n_before - n_after

    print(f"[STATS] Total: {n_before}  Kept (<= {MAX_WETLAND_AREA_M2:,} m²): {n_after}  Removed: {n_removed}")

    # Write outputs (Shapefile + optional GeoPackage)
    print(f"[WRITE] Shapefile → {OUT_SHP}")
    kept.to_file(OUT_SHP)

    print(f"[WRITE] GeoPackage → {OUT_GPKG} (layer='wetlands')")
    kept.to_file(OUT_GPKG, layer="wetlands", driver="GPKG")

    print("[DONE] Filtering complete.")

if __name__ == "__main__":
    main()


[LOAD] /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/working/DML_wetlands_from_Tyler_clipped_5070.shp
[STATS] Total: 200470  Kept (<= 647,500 m²): 200386  Removed: 84
[WRITE] Shapefile → /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/Historical_Wetlands_FILT/Historical_Wetlands_TYLER_max647500m2_5070.shp
[WRITE] GeoPackage → /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/Historical_Wetlands_FILT/Historical_Wetlands_TYLER_max647500m2_5070.gpkg (layer='wetlands')
[DONE] Filtering complete.
